# ABSA — chỉ dùng train.csv

Không đọc hoặc dự đoán blind; không tạo file nộp. Đây là notebook nghiên cứu validation, chưa là bộ nộp hoàn chỉnh. Mỗi phương pháp có thư mục riêng. Chỉ so sánh trực tiếp các lượt cùng split. Validation đã được xem trong nhiều lượt phát triển; không coi đây là test độc lập.

PhoBERT trong các lượt mới là encoder đóng băng int8, học classifier/attention head. Chưa fine-tune toàn bộ encoder. Cell stacking chạy lại meta từ đầu ra nguồn đã lưu, không huấn luyện lại encoder.


In [1]:
import train_only_guard
import json,time
from pathlib import Path
import numpy as np,pandas as pd,joblib
from scipy import sparse
from absa_core import LABELS,reference_splits,prepare_texts,binary_predictions
from aspect_scope import scoped_texts
from score_absa import report,show
ROOT=Path.cwd()
train=pd.read_csv(ROOT/'train.csv')
assert list(train.columns)==['id','text']+LABELS
fit,tune,valid=reference_splits(train)
print('Train:',len(train),'fit:',len(fit),'tune:',len(tune),'validation:',len(valid))
print('Blind CSV audit guard active.')


Train: 4688 fit: 2812 tune: 938 validation: 938
Blind CSV audit guard active.


In [2]:
rows=[]
for path in sorted(ROOT.glob('*/metrics.json')):
    m=json.loads(path.read_text())
    policy=m.get('split_policy','legacy_random42')
    if path.parent.name=='Ensemble_lan2':policy='reference_group_2026'
    rows.append({'experiment':path.parent.name,'split':policy,'ABSA_F1':m['ABSA_F1'],
        'CI95':m.get('ABSA_F1_CI95',None),'accuracy':m.get('accuracy_TB',m.get('accuracy_mean_per_label'))})
table=pd.DataFrame(rows)
print('Completed measured runs:',len(table))
display(table[table.split=='reference_group_2026'].sort_values('ABSA_F1',ascending=False))
print('Target 0.96 reached:',bool((table.ABSA_F1>=.96).any()))


Completed measured runs: 46


,experiment,split,ABSA_F1,CI95,accuracy
29,Stacking_lan1,reference_group_2026,0.799716,"[0.7823207024217709, 0.8139899961823089]",0.941288
0,AugmentScoped_lan1,reference_group_2026,0.792965,"[0.7747530970587293, 0.8085608804270658]",0.934816
27,ScopedTFIDF_lan4,reference_group_2026,0.787288,"[0.7696595613021269, 0.8025069354372548]",0.934892
28,ScopedTFIDF_lan5,reference_group_2026,0.785876,"[0.7685836506460388, 0.8022491081214727]",0.934663
1,AugmentScoped_lan2,reference_group_2026,0.779123,"[0.7609165626076664, 0.7943751101878496]",0.932227
24,ScopedTFIDF_lan1,reference_group_2026,0.778488,"[0.7605036565026525, 0.7952951287161931]",0.930627
13,PhoBERT_Attention_lan2,reference_group_2026,0.777668,"[0.760718755319747, 0.7931073194626393]",0.924535
25,ScopedTFIDF_lan2,reference_group_2026,0.772734,"[0.7559417497271771, 0.7884780979322252]",0.929485
18,PhoBERT_TFIDF_lan2,reference_group_2026,0.769551,"[0.753046019179463, 0.7849429821630949]",0.919053
16,PhoBERT_RDR_lan3,reference_group_2026,0.766883,"[0.7483672245561196, 0.7827665632257274]",0.922175


Target 0.96 reached: False


## Tái chạy toàn bộ suy luận của mô hình theo mệnh đề

Mô hình validation chỉ học outer80. Tách ngưỡng trên tune trước khi refit. Các lexical scope chọn đầu vào; nhãn cuối do mô hình học từ train, không phải quy tắc gán nhãn.


In [3]:
folder=ROOT/'ScopedTFIDF_lan4'
bundle=joblib.load(folder/'validation_model.joblib')
texts=np.asarray(prepare_texts(train.text.iloc[valid],'lexical'),dtype=object)
scopes=scoped_texts(train.text.iloc[valid])
started=time.perf_counter();G=bundle['global_vectorizer'].transform(texts);parts=[]
for aspect,vec,model in zip(['nhan_vien','sach_se','vi_tri','phong','gia_tri','an_uong','tien_nghi'],bundle['local_vectorizers'],bundle['models']):
    L=vec.transform(scopes[aspect])
    parts.append(model.decision_function(sparse.hstack([G,L*bundle['config']['scope_weight']],format='csr')))
scores=np.hstack(parts);pred=binary_predictions(scores,bundle['thresholds'])
gt=train.iloc[valid][['id']+LABELS].reset_index(drop=True)
sb=pd.DataFrame(pred,columns=LABELS);sb.insert(0,'id',gt.id)
metrics=report(gt,sb,n_boot=1000);show(metrics)
assert abs(metrics['ABSA_F1']-json.loads((folder/'metrics.json').read_text())['ABSA_F1'])<1e-12
print('Validation model inference seconds:',round(time.perf_counter()-started,4))


Số mẫu: 938
ABSA-F1 (điểm xếp hạng) = 0.787   CI95% [0.770, 0.803]
F1 theo cực: khen 0.865 | chê 0.709 | trung tính (phụ) 0.012 | phát hiện khía cạnh (phụ) 0.896
Nhãn yếu nhất: vi_tri_che F1 = 0.585; độ lệch chuẩn F1 14 nhãn = 0.119
Precision TB = 0.793 | Recall TB = 0.784 | Accuracy TB = 0.935 (14 nhãn)
  khen: P 0.874 R 0.858 | chê: P 0.712 R 0.709
kappa TB = 0.744 | MCC TB = 0.745 | Balanced acc TB = 0.870
F1 vi mô: khía cạnh = 0.909 | (khía cạnh, cực) = 0.825 | khớp đúng cả đánh giá = 0.453
Mốc lớp đa số =   nan; cải thiện tương đối =   nan

Khía cạnh      Nhắc F1 | Khen F1     P     R     κ     n | Chê F1      P     R     κ     n | Trung tính F1  n
nhan_vien      0.946  | 0.940 0.950 0.929 0.874   495 | 0.758  0.761 0.755 0.726   110 | 0.000       0
sach_se        0.952  | 0.971 0.971 0.971 0.957   310 | 0.750  0.794 0.711 0.729    76 | 0.000       0
vi_tri         0.923  | 0.918 0.926 0.911 0.878   315 | 0.585  0.596 0.574 0.560    54 | 0.000       2
phong          0.923  | 0.848

## Tái tính stacking từ đầu ra encoder/classifier đã lưu

Không coi thời gian cell này là thời gian huấn luyện hay chạy lại encoder. Base neural heads đã chọn epoch trên tune; meta CV không phải nested encoder CV hoàn toàn. Validation vẫn ngoài mọi bước fit/chọn ngưỡng của mô hình này.


In [4]:
folder=ROOT/'Stacking_lan1'
bundle=joblib.load(folder/'meta_models.joblib')
sources=[np.load(ROOT/name/'private_validation_scores.npy') for name in bundle['sources']]
scores=[]
for j,(model,cfg) in enumerate(zip(bundle['models'],bundle['labels'])):
    idx=list(range(21)) if cfg['mode']=='all' else list(range((j//3)*3,(j//3+1)*3))
    scores.append(model.decision_function(np.concatenate([s[:,idx] for s in sources],axis=1)))
scores=np.column_stack(scores);pred=binary_predictions(scores,np.asarray(bundle['thresholds']))
sb=pd.DataFrame(pred,columns=LABELS);sb.insert(0,'id',gt.id)
metrics=report(gt,sb,n_boot=1000);show(metrics)
assert abs(metrics['ABSA_F1']-json.loads((folder/'metrics.json').read_text())['ABSA_F1'])<1e-12
print('Target 0.96 reached:',metrics['ABSA_F1']>=.96)


Số mẫu: 938
ABSA-F1 (điểm xếp hạng) = 0.800   CI95% [0.782, 0.814]
F1 theo cực: khen 0.879 | chê 0.720 | trung tính (phụ) 0.008 | phát hiện khía cạnh (phụ) 0.912
Nhãn yếu nhất: vi_tri_che F1 = 0.574; độ lệch chuẩn F1 14 nhãn = 0.121
Precision TB = 0.829 | Recall TB = 0.777 | Accuracy TB = 0.941 (14 nhãn)
  khen: P 0.892 R 0.868 | chê: P 0.767 R 0.686
kappa TB = 0.761 | MCC TB = 0.763 | Balanced acc TB = 0.872
F1 vi mô: khía cạnh = 0.921 | (khía cạnh, cực) = 0.825 | khớp đúng cả đánh giá = 0.463
Mốc lớp đa số =   nan; cải thiện tương đối =   nan

Khía cạnh      Nhắc F1 | Khen F1     P     R     κ     n | Chê F1      P     R     κ     n | Trung tính F1  n
nhan_vien      0.950  | 0.940 0.962 0.919 0.876   495 | 0.773  0.825 0.727 0.745   110 | 0.000       0
sach_se        0.967  | 0.982 0.981 0.984 0.974   310 | 0.727  0.857 0.632 0.707    76 | 0.000       0
vi_tri         0.926  | 0.935 0.957 0.914 0.903   315 | 0.574  0.617 0.537 0.550    54 | 0.000       2
phong          0.925  | 0.867

In [5]:
# Chỉ lưu lỗi từ validation của TRAIN; không đưa văn bản lên output công khai.
errors=[]
truth=gt[LABELS].to_numpy(int)
for k in np.flatnonzero((truth!=pred).any(axis=1))[:5]:
    fp=[LABELS[j] for j in np.flatnonzero((pred[k]==1)&(truth[k]==0))]
    fn=[LABELS[j] for j in np.flatnonzero((pred[k]==0)&(truth[k]==1))]
    errors.append({'id':str(gt.id.iloc[k]),'text':str(train.text.iloc[valid[k]]),
        'false_positive':fp,'false_negative':fn,
        'review_note':'Đối chiếu phạm vi khía cạnh và phủ định/gợi ý cải thiện; phân biệt nhắc với cảm xúc. Đây là phân tích lỗi train, không sửa nhãn hoặc dự đoán blind.'})
(folder/'private_error_analysis.json').write_text(json.dumps(errors,ensure_ascii=False,indent=2))
print('Private train-validation error examples saved:',len(errors))


Private train-validation error examples saved: 5


In [6]:
provenance=ROOT/'augmentation_provenance.json'
if provenance.exists():
    p=json.loads(provenance.read_text())
    print('Synthetic training rows:',p['rows'])
    print('Tool:',p['tool'])
    print('User prompt:',p['user_prompt'])
    print('Construction specification:',p['construction_spec'])
    print('SHA256:',p['sha256'])
    print('If selecting an augmented model, include du_lieu_them.csv and provenance in final zip.')
    if (ROOT/'augmentation_comparison.csv').exists():display(pd.read_csv(ROOT/'augmentation_comparison.csv'))
print('TRAIN-ONLY: no submission.csv generated, no blind CSV opened.')


Synthetic training rows: 640
Tool: ChatGPT/Codex authored templates and labels; Python deterministic combinations; no external labeling API
User prompt: do k được dùng file test_blind á nên chạy làm sao cho được 0.96 đi bạn tôi tiền xử lý dữ kiệu, phobert tfidf mà được trên 0.9 đó. tìm cách đi
Construction specification: Create novel Vietnamese review clauses for the assignment seven aspects; label mention/praise/complaint including mild complaint, temporary acceptance and improvement requests. Combine clauses deterministically. Never use blind text or original review retrieval.
SHA256: 10d53ce8c262710a98a7b5b337b88fcf509c48ddbfb3c1cc358d510af9b7c3b4
If selecting an augmented model, include du_lieu_them.csv and provenance in final zip.


,experiment,ABSA_F1,baseline,baseline_ABSA_F1,delta_ABSA_F1,CI95_low,CI95_high
0,AugmentScoped_lan1,0.792965,ScopedTFIDF_lan5,0.785876,0.007089,0.774753,0.808561
1,AugmentScoped_lan2,0.779123,ScopedTFIDF_lan4,0.787288,-0.008166,0.760917,0.794375


TRAIN-ONLY: no submission.csv generated, no blind CSV opened.
